# Image Motion Studio — GPU Worker

This notebook turns a free Google Colab session into the Image Motion Studio GPU worker. It:

1. Verifies the GPU and system RAM
2. Sets your **CONVEX_HTTP_BASE**, **WORKER_TOKEN** and (optionally) **MODEL_ID**
3. Installs dependencies and loads **WAN 2.2 TI2V-5B** (diffusers, 4-bit quantized — T4-safe)
4. Polls the studio for queued image-to-video jobs
5. Renders each job on the GPU and uploads the MP4 straight to Convex storage

Before running: issue a worker token in the studio (Settings → Worker token → Issue token) and paste it below together with your Convex HTTP base URL (Settings shows both values).

In [ ]:
#@title 1 · Configuration — paste your two studio values here

# From the studio Settings page → "Colab worker setup":
CONVEX_HTTP_BASE = ""  #@param {type:"string"}
WORKER_TOKEN = ""             #@param {type:"string"}

# Model to load. Leave as-is for the recommended free-Colab model.
# This is the only WAN image-to-video model that fits a free T4 when loaded
# 4-bit (cell 4). A14B/14B models are blocked — they need A100-class VRAM.
MODEL_ID = "wan2.2-ti2v-5b"

assert CONVEX_HTTP_BASE.startswith("https://"), "Set CONVEX_HTTP_BASE to your .convex.site URL"
assert len(WORKER_TOKEN) >= 16, "Paste the token issued in Settings (32+ chars)"

import requests, json, time

def api(path, payload):
    r = requests.post(f"{CONVEX_HTTP_BASE}/worker_api/{path}", json={"token": WORKER_TOKEN, **payload}, timeout=120)
    r.raise_for_status()
    return r.json()

# First contact — wrong token or wrong URL fails here, loudly.
health = api("health", {"status": "online", "message": "Colab worker starting"})
print("✓ authenticated —", CONVEX_HTTP_BASE, json.dumps(health))

✓ authenticated — https://formal-kookabura-53.convex.site {"ok": true, "serverTime": 1789627947914}


In [ ]:
#@title 2 · GPU + system RAM check
import torch, psutil

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime → Change runtime type → select a GPU (free T4 is fine).")

GPU_NAME = torch.cuda.get_device_name(0)
FREE, TOTAL = torch.cuda.mem_get_info()
VRAM_GB = round(TOTAL / 1024**3, 1)
RAM_TOTAL_GB = round(psutil.virtual_memory().total / 1024**3, 1)
print(f"GPU: {GPU_NAME} · {VRAM_GB} GB VRAM · {round(FREE/1024**3,1)} GB free")
print(f"System RAM: {RAM_TOTAL_GB} GB")

api("health", {"status": "online", "gpuName": GPU_NAME, "vramGb": VRAM_GB, "message": "GPU ready"})
print("✓ GPU reported to the studio")

GPU: Tesla T4 · 14.6 GB VRAM · 14.5 GB free
System RAM: 12.7 GB
✓ GPU reported to the studio


In [ ]:
#@title 3 · Install dependencies (first run only, ~3–5 minutes)
# Version floors: WanPipeline needs diffusers ≥ 0.35, the Wan/UMT5 model
# classes need transformers ≥ 4.55, 4-bit bnb loading needs bitsandbytes ≥ 0.45.
%pip -q install -U "diffusers>=0.35.1" "transformers>=4.55.4" "accelerate>=1.1.0" "bitsandbytes>=0.45.0" imageio[ffmpeg] safetensors
import diffusers, transformers, bitsandbytes
print("diffusers", diffusers.__version__, "· transformers", transformers.__version__, "· bitsandbytes", bitsandbytes.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.1 MB/s eta 0:00:00
diffusers 0.40.0 · transformers 5.17.0 · bitsandbytes 0.50.2


In [ ]:
#@title 3 · Load WAN 2.2 TI2V-5B — T4-safe Image → Video
import gc
import torch

from diffusers import WanImageToVideoPipeline, WanTransformer3DModel, AutoencoderKLWan
from transformers import UMT5EncoderModel, BitsAndBytesConfig

MODEL_REPOS = {
    "wan2.2-ti2v-5b": "Wan-AI/Wan2.2-TI2V-5B-Diffusers",
}

if MODEL_ID not in MODEL_REPOS:
    raise ValueError(
        f"{MODEL_ID} cannot run on a free Colab T4. "
        f"Use {list(MODEL_REPOS)}."
    )

MODEL_REPO = MODEL_REPOS[MODEL_ID]
DTYPE = torch.float16

# 4-bit NF4 is used to keep the text encoder + transformer within T4 VRAM.
quant_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=DTYPE,
)

MAX_MEMORY = {
    0: "11GiB",
    "cpu": "10GiB",
}

print("· 1/3 text encoder UMT5 (4-bit, auto placement)…")
text_encoder = UMT5EncoderModel.from_pretrained(
    MODEL_REPO,
    subfolder="text_encoder",
    quantization_config=quant_4bit,
    torch_dtype=DTYPE,
    device_map="auto",
    max_memory=MAX_MEMORY,
    low_cpu_mem_usage=True,
)
print("✓ UMT5 loaded")
print("  device map:", getattr(text_encoder, "hf_device_map", "not available"))

gc.collect()
torch.cuda.empty_cache()

print("· 2/3 transformer (4-bit, auto placement)…")
transformer = WanTransformer3DModel.from_pretrained(
    MODEL_REPO,
    subfolder="transformer",
    quantization_config=quant_4bit,
    torch_dtype=DTYPE,
    device_map="auto",
    max_memory=MAX_MEMORY,
    low_cpu_mem_usage=True,
)
print("✓ transformer loaded")
print("  device map:", getattr(transformer, "hf_device_map", "not available"))

gc.collect()
torch.cuda.empty_cache()

print("· 3/3 VAE (fp32)…")
vae = AutoencoderKLWan.from_pretrained(
    MODEL_REPO,
    subfolder="vae",
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True,
)
vae = vae.to("cuda")
print("✓ VAE loaded")

print("· assembling WanImageToVideoPipeline…")
pipe = WanImageToVideoPipeline.from_pretrained(
    MODEL_REPO,
    transformer=transformer,
    text_encoder=text_encoder,
    vae=vae,
    torch_dtype=DTYPE,
)

gc.collect()
torch.cuda.empty_cache()

print("· pipeline diagnostics")
print("  diffusers:", __import__("diffusers").__version__)
print("  pipeline:", type(pipe).__name__)
print("  image-to-video call supports image:",
      "image" in __import__("inspect").signature(pipe.__call__).parameters)

if "image" not in __import__("inspect").signature(pipe.__call__).parameters:
    raise RuntimeError(
        "Wrong Wan pipeline loaded: Image → Video requires a pipeline "
        "whose __call__ accepts image=."
    )

allocated_gb = torch.cuda.memory_allocated() / (1024 ** 3)
reserved_gb = torch.cuda.memory_reserved() / (1024 ** 3)

print(
    f"✓ model loaded — "
    f"VRAM allocated: {allocated_gb:.1f} GB | "
    f"reserved: {reserved_gb:.1f} GB"
)

api(
    "health",
    {
        "status": "online",
        "gpuName": GPU_NAME,
        "vramGb": VRAM_GB,
        "loadedModel": MODEL_ID + " (4-bit T4, Image → Video)",
        "message": "Model loaded — polling for jobs",
    },
)

print("✓ TI2V-5B Image → Video worker ready")


· 1/3 text encoder UMT5 (4-bit, auto placement)…


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/22.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

model_index.json:   0%|          | 0.00/499 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

✓ model loaded — VRAM allocated: 12.5 GB | reserved: 12.7 GB
✓ model loaded and reported to the studio


In [ ]:
#@title 5 · Worker loop — claims jobs, renders, uploads MP4s. Keep this cell running.
import io, time, torch, imageio
import PIL.Image
from PIL import Image

HEARTBEAT_SECONDS = 60
CLAIM_INTERVAL_SECONDS = 15

def report_progress(key, status, progress, message):
    try:
        api("progress", {"workerJobKey": key, "status": status,
                         "progress": progress, "message": message})
    except Exception as e:
        print("  (progress update failed, continuing):", e)

def check_cancelled(key):
    try:
        out = api("job-status", {"workerJobKey": key})
        return bool(out.get("cancelled"))
    except Exception:
        return False

def upload_bytes(upload_url, data: bytes, content_type: str) -> str:
    r = requests.post(upload_url, data=data,
                      headers={"Content-Type": content_type}, timeout=600)
    r.raise_for_status()
    return r.json()["storageId"]

# Aspect-ratio aware frame sizing (WAN TI2V-5B needs multiples of 32: VAE 16 x patch 2).
def frame_size(aspect: str, resolution: str):
    short = 480 if resolution == "480p" else 720
    def snap(n): return int(round(n / 32)) * 32  # 32 = VAE 16 x transformer patch 2
    if aspect == "9:16":  return snap(short), snap(short * 16 / 9)
    if aspect == "1:1":   return snap(short), snap(short)
    return snap(short * 16 / 9), snap(short)  # 16:9 default

def render_job(job) -> bytes:
    """Runs REAL image-to-video inference. No placeholders, no sample output."""
    s = job["settings"]
    # Free-T4 VRAM guard: 720p exhausts 15 GB VRAM on this runtime. 480p keeps
    # renders stable; the MP4 output itself is unchanged.
    resolution = s.get("resolution", "480p")
    if resolution == "720p":
        print("  (720p requested — clamped to 480p for free-T4 VRAM)")
        resolution = "480p"
    width, height = frame_size(s.get("aspectRatio", "16:9"), resolution)

    # 1 · download the source image and make it the first frame
    img = Image.open(io.BytesIO(requests.get(job["imageUrl"], timeout=120).content)).convert("RGB")
    img = img.resize((width, height), Image.LANCZOS)

    # 2 · video generation (this is the actual GPU work)
    # pipe is WanImageToVideoPipeline, whose API explicitly accepts image=.
    generator = torch.Generator("cuda")
    if s.get("seed") is not None:
        generator = generator.manual_seed(int(s["seed"]))

    steps = 20                      # TI2V-5B quality/runtime trade-off for Colab
    total = steps + 12              # + encode/upload bookkeeping for progress math
    done = 0
    def bump(p, msg):
        nonlocal done
        done = p
        report_progress(job["workerJobKey"], "generating",
                        min(95, round(p / total * 100)), msg)

    report_progress(job["workerJobKey"], "loading_model", 2, "preparing tensors")

    result = pipe(
        image=img,
        prompt=job["prompt"],
        negative_prompt=job.get("negativePrompt") or None,
        width=width,
        height=height,
        num_frames=81,              # TI2V-5B native clip length (~5 s at 16 fps)
        num_inference_steps=steps,
        guidance_scale=5.0,
        generator=generator,
        callback_on_step_end=lambda p, i, t, kw: (bump(i + 1, f"diffusion step {i+1}/{steps}"), kw)[1],
    )
    frames = result.frames[0]       # list[PIL.Image]

    # 3 · encode a real MP4 with ffmpeg via imageio
    report_progress(job["workerJobKey"], "processing", 90, "encoding MP4")
    buf = io.BytesIO()
    imageio.mimwrite(buf, frames, format="FFMPEG", codec="libx264",
                     fps=16, quality=8, pixelformat="yuv420p",
                     macro_block_size=16, output_params=["-movflags", "+faststart"])
    torch.cuda.empty_cache()        # release inference buffers before the next job
    return buf.getvalue()

print("worker running — claim/heartbeat loop started")
print("queue a job in the studio; this cell picks it up automatically")
print("stop the cell (⏹) or Runtime → Interrupt to take the worker offline")

last_heartbeat = 0.0
while True:
    now = time.time()
    try:
        if now - last_heartbeat > HEARTBEAT_SECONDS:
            api("health", {"status": "online", "gpuName": GPU_NAME, "vramGb": VRAM_GB,
                           "loadedModel": MODEL_ID + " (4-bit T4)", "message": "polling for jobs"})
            last_heartbeat = now

        claim = api("claim", {})
        job = claim.get("job")
        if not job:
            time.sleep(CLAIM_INTERVAL_SECONDS)
            continue

        print("▸ job", job["workerJobKey"][:8], "·", job["model"])
        report_progress(job["workerJobKey"], "connecting", 1, "claimed — downloading source image")

        if check_cancelled(job["workerJobKey"]):
            print("  job was cancelled before render — skipping")
            continue

        try:
            mp4 = render_job(job)

            if check_cancelled(job["workerJobKey"]):
                print("  job was cancelled during render — discarding")
                continue

            video_storage_id = upload_bytes(job["videoUploadUrl"], mp4, "video/mp4")
            out = api("complete", {"workerJobKey": job["workerJobKey"],
                                   "videoStorageId": video_storage_id})
            print("  ✓ completed — videoId", out.get("videoId", "?"))

        except KeyboardInterrupt:
            raise
        except Exception as e:
            msg = str(e)[:500]
            print("  ✗ job failed:", msg)
            try:
                api("fail", {"workerJobKey": job["workerJobKey"], "error": msg})
            except Exception as e2:
                print("  (fail report also failed):", e2)

    except KeyboardInterrupt:
        print("worker stopped by user")
        try:
            api("health", {"status": "error", "message": "worker stopped"})
        except Exception:
            pass
        break
    except Exception as e:
        # transient network errors: wait and retry, the studio shows truth
        print("loop error (retrying):", str(e)[:200])
        time.sleep(10)

worker running — claim/heartbeat loop started
queue a job in the studio; this cell picks it up automatically
stop the cell (⏹) or Runtime → Interrupt to take the worker offline
▸ job e0ba5ef4 · wan2.2-i2v-a14b
  ✗ job failed: WanPipeline.__call__() got an unexpected keyword argument 'image'
